# Demonstracja ETL: Dane przed i po
Poniższy notatnik pozwala przetestować na żywo, jak dane wyglądały przed wyczyszczeniem (w strefie Staging) oraz jak wyglądają w hurtowni (Data Warehouse) po transformacji.

In [1]:
import os
import sys
# Dodajemy ścieżkę projektu do sys.path, aby móc importować moduły z src
sys.path.append(os.path.abspath('..'))

import pandas as pd
from src.utils.db import sqlserver_connection

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("Zależności załadowane pomyślnie!")

Zależności załadowane pomyślnie!


## Funkcja pomocnicza do zapytań
Przygotujmy prostą funkcję, która pobierze dane z SQL Server do obiektu DataFrame.

In [2]:
def query_db(sql_query: str) -> pd.DataFrame:
    with sqlserver_connection() as conn:
        return pd.read_sql(sql_query, conn)

## Krok 1: Dane Surowe (Staging) - "Przed"
Sprawdźmy, jak wyglądają surowe dane przed zastosowaniem procesów ETL.

In [3]:
staging_sql = """
SELECT TOP 5 
    invoice_and_item_number,
    date,
    store_location,      -- Format przestrzenny POINT
    category_name,       -- Zwróć uwagę na braki danych (NULL)
    state_bottle_cost,   -- Format z symbolem waluty $
    sale_dollars         -- Format z symbolem waluty $
FROM stg.iowa_liquor_sales_raw;
"""
df_stg = query_db(staging_sql)
df_stg.head()

/tmp/ipykernel_886/3541370584.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql_query, conn)


,invoice_and_item_number,date,store_location,category_name,state_bottle_cost,sale_dollars
0,INV-54554000001,2023-01-02,POINT (-93.61378 41.60575),100% AGAVE TEQUILA,14.50,261.00
1,INV-54554000002,2023-01-02,POINT (-93.61378 41.60575),AMERICAN VODKAS,4.65,418.80
2,INV-54554000003,2023-01-02,POINT (-93.61378 41.60575),IMPORTED FLAVORED VODKA,9.96,358.56
3,INV-54554000004,2023-01-02,POINT (-93.61378 41.60575),CREAM LIQUEURS,17.00,306.00
4,INV-54554000005,2023-01-02,POINT (-93.61378 41.60575),SPICED RUM,12.49,1124.40


## Krok 2: Oczyszczone wymiary i fakty - "Po"
Zobaczmy jak dane zostały przetransformowane do czystej i spójnej postaci w schemacie `dw` (Data Warehouse).

In [4]:
store_sql = """
SELECT TOP 5 
    store_key, 
    store_name, 
    latitude, 
    longitude 
FROM dw.dim_store;
"""
df_store = query_db(store_sql)
print("Wyodrębnione współrzędne w wymiarze sklepu:")
display(df_store)

Wyodrębnione współrzędne w wymiarze sklepu:


/tmp/ipykernel_886/3541370584.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql_query, conn)


,store_key,store_name,latitude,longitude
0,1,JACK & JILL STORE / WEST BRANCH,41.670494,-91.343413
1,2,LOCAL LIQUOR / PANORA,41.692594,-94.357208
2,3,LEGENDARY RYE / BAD BEAR ENTERPRISES (ET),42.067186,-94.866874
3,4,KWIK STAR #1158 / AMES,42.008984,-93.587699
4,5,EMPIRE LIQUOR AND TOBACCO / HIAWATHA,42.046042,-91.673047


In [5]:
fact_sql = """
SELECT TOP 5 
    invoice_number,
    sale_dollars,       -- Teraz to czysty DECIMAL
    state_bottle_cost,  -- Teraz to czysty DECIMAL
    bottles_sold,
    margin_amount       -- Wyliczona nowa miara pochodna
FROM dw.fact_sales;
"""
df_fact = query_db(fact_sql)
print("Oczyszczone finanse i wyliczona marża w tabeli faktów:")
display(df_fact)

Oczyszczone finanse i wyliczona marża w tabeli faktów:


/tmp/ipykernel_886/3541370584.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql_query, conn)


,invoice_number,sale_dollars,state_bottle_cost,bottles_sold,margin_amount
0,INV-54554000001,261.00,14.50,12.0,87.00
1,INV-54554000002,418.80,4.65,60.0,139.80
2,INV-54554000003,358.56,9.96,24.0,119.52
3,INV-54554000004,306.00,17.00,12.0,102.00
4,INV-54554000005,1124.40,12.49,60.0,375.00


### Obsługa braków danych
Zobaczmy, czy brakujące przypisania kategorii (które w stg były puste lub NULL) zostały obsłużone za pomocą rekordu `UNKNOWN`.

In [6]:
category_sql = """
SELECT * 
FROM dw.dim_category
WHERE category_number = 'UNKNOWN';
"""
df_category = query_db(category_sql)
display(df_category)

/tmp/ipykernel_886/3541370584.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql_query, conn)


,category_key,category_number,category_name


## Krok 3: Warstwa semantyczna - Gotowe raporty
Ostatecznie nasza aplikacja odpytuje tylko bezpieczne widoki semantyczne (schemat `sem`), które hermetyzują logikę i dają czyste wyniki dla biznesu.

In [7]:
semantic_sql = """
SELECT TOP 10 * 
FROM sem.vw_sales_by_month
ORDER BY year, month;
"""
df_sem = query_db(semantic_sql)
display(df_sem)

/tmp/ipykernel_886/3541370584.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql_query, conn)


,year,quarter,month,year_month,total_sales,total_bottles_sold,total_volume_liters,total_margin,sales_line_count,invoice_count,store_count
0,2023,1,1,2023-01,32582340.63,2358449.0,1747102.70,10888433.83,212850,212850,1857
1,2023,1,2,2023-02,32134462.65,2289374.0,1771119.28,10750429.35,189294,189294,1824
2,2023,1,3,2023-03,36436060.72,2632521.0,2016226.87,12184903.75,221313,221313,1859
3,2023,2,4,2023-04,32915910.22,2393665.0,1793682.48,10986201.79,198446,198446,1832
4,2023,2,5,2023-05,39721449.29,2786013.0,2187271.06,13398186.64,233060,233060,1879
5,2023,2,6,2023-06,41269784.93,2914814.0,2221920.35,13754934.75,247634,247634,1902
6,2023,3,7,2023-07,35899983.05,2494141.0,1870925.05,11983280.37,208841,208841,1873
7,2023,3,8,2023-08,39843169.23,2780638.0,2130832.53,13299651.21,230068,230068,1899
8,2023,3,9,2023-09,35140021.42,2491143.0,1883281.17,11738767.46,208937,208937,1891
9,2023,4,10,2023-10,40330287.00,2744859.0,2038313.05,13455110.03,220976,220976,1883
